# Geospatial EDA of the EV charging stations

This notebook showcases the analysis implemented in `src/eda/geospatial.py` (geopandas + folium, no API keys needed): where the stations are, how heavily they are used, and how usage relates to location.

In [1]:
import os
# run from the project root so relative data paths and `src` imports work
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import warnings
warnings.filterwarnings("ignore")

from src.eda import geospatial as geo

## 1. Data
The cleaned dataset from `src/clean_data.py` (rebuilt from the raw data if the CSV is missing).

In [2]:
df = geo.load_data()
print(f"{len(df):,} sessions")
df[["Station Name", "Latitude", "Longitude", "year", "total_duration_hours", "idle_hours", "Energy (kWh)"]].head()

Dropped 249 rows with invalid or missing dates.


Dropped 120 rows where start and end time zones are different.


Removed 45 rows with inconsistent durations, remaining: 259001


Removed 308 rows (0.12%), remaining: 258693
Final size: 258693. Dropped in total: 722 rows (0.28%).
258,693 sessions


,Station Name,Latitude,Longitude,year,total_duration_hours,idle_hours,Energy (kWh)
0,PALO ALTO CA / HAMILTON #1,37.444572,-122.160309,2011,3.058889,1.158056,6.249457
1,PALO ALTO CA / HAMILTON #1,37.444572,-122.160309,2011,0.035000,0.003333,0.106588
2,PALO ALTO CA / HAMILTON #1,37.444572,-122.160309,2011,4.292222,0.001111,14.951777
3,PALO ALTO CA / HAMILTON #1,37.444572,-122.160309,2011,2.056667,0.007222,7.159643
4,PALO ALTO CA / HAMILTON #1,37.444572,-122.160309,2011,1.190000,0.458333,1.957765


## 2. Station table
One row per station. Some stations report slightly different coordinates over time, so the location is the **median** of the reported coordinates. The result is a `GeoDataFrame` with point geometries (WGS84).

In [3]:
stations = geo.build_station_gdf(df)
print(stations.crs, "-", len(stations), "stations")
stations.sort_values("n_events", ascending=False).drop(columns=["geometry", "Station Name"]).head(10).round(2)

EPSG:4326 - 47 stations


,lat,lon,address,n_events,n_ports,avg_total_hours,avg_charging_hours,avg_idle_hours,pct_long_idle,total_energy_kwh,avg_energy_kwh,short_name
13,37.44,-122.16,250 Hamilton Ave,23659,2,2.51,2.03,0.48,26.47,191072.87,8.08,HAMILTON #2
12,37.44,-122.16,250 Hamilton Ave,14830,2,2.52,2.04,0.48,28.26,113328.92,7.64,HAMILTON #1
44,37.45,-122.16,500-532 Webster St,14394,2,2.76,2.15,0.61,29.85,138631.99,9.63,WEBSTER #1
2,37.45,-122.16,Bryant St,14293,3,2.54,2.00,0.54,25.87,106628.44,7.46,BRYANT #2
17,37.44,-122.16,528 High St,13835,3,2.36,1.95,0.41,26.26,111371.38,8.05,HIGH #4
1,37.45,-122.16,Bryant St,13391,3,2.60,2.07,0.53,26.38,101952.79,7.61,BRYANT #1
45,37.45,-122.16,500-532 Webster St,12624,2,2.81,2.16,0.64,32.05,125326.51,9.93,WEBSTER #2
7,37.43,-122.15,400 Cambridge Ave,11147,2,2.28,1.88,0.40,24.92,80367.22,7.21,CAMBRIDGE #1
14,37.44,-122.16,528 High St,10735,2,2.38,1.93,0.45,26.37,98248.19,9.15,HIGH #1
46,37.45,-122.16,520 Webster St,10459,2,2.66,2.09,0.57,30.19,98326.08,9.40,WEBSTER #3


## 3. Spatial features
Distances are computed in a metric projection (UTM zone 10N), so they are in metres:
- `nearest_station_m` - distance to the closest other station
- `n_within_250m` - number of other stations within 250 m
- `dist_to_center_m` - distance to the usage-weighted centre of all stations

In [4]:
stations = geo.add_spatial_features(stations)
stations.sort_values("n_events", ascending=False)[
    ["short_name", "n_events", "nearest_station_m", "n_within_250m", "dist_to_center_m"]
].head(10).round(1)

,short_name,n_events,nearest_station_m,n_within_250m,dist_to_center_m
13,HAMILTON #2,23659,16.2,1,973.3
12,HAMILTON #1,14830,16.2,3,973.7
44,WEBSTER #1,14394,3.6,2,1149.8
2,BRYANT #2,14293,8.6,6,1233.1
17,HIGH #4,13835,13.4,4,1124.4
1,BRYANT #1,13391,10.9,6,1251.8
45,WEBSTER #2,12624,4.8,2,1142.2
7,CAMBRIDGE #1,11147,3.0,13,1556.2
14,HIGH #1,10735,13.4,4,1116.6
46,WEBSTER #3,10459,3.6,2,1146.9


### Does usage depend on location?

In [5]:
for col in ["dist_to_center_m", "nearest_station_m"]:
    rho = stations[[col, "n_events"]].corr(method="spearman").iloc[0, 1]
    print(f"Spearman correlation of n_events with {col}: {rho:.2f}")

Spearman correlation of n_events with dist_to_center_m: -0.63
Spearman correlation of n_events with nearest_station_m: 0.05


Usage drops clearly with distance from the centre of activity, whereas the distance to the nearest other station is unrelated to usage. Several stations share practically the same spot (a few metres apart), so they are clusters of ports rather than separate sites.

## 4. Interactive map
Use the layer control (top right) to switch between:
- number of events per station (marker size and colour)
- share of sessions with idle time > 20 min
- average energy per session
- heatmap of charging events
- one layer per year

Hover over a station for details. The star marks the usage-weighted centre.

In [6]:
m = geo.build_map(df, stations)
m

## 5. Saving the outputs

In [7]:
os.makedirs(geo.OUT_DIR, exist_ok=True)
m.save(os.path.join(geo.OUT_DIR, "station_map.html"))
stations.to_file(os.path.join(geo.OUT_DIR, "stations.geojson"), driver="GeoJSON")
os.listdir(geo.OUT_DIR)

['stations.geojson', 'station_map.html', 'station_summary.csv']